# inps-pensioni — notebook v0

Notebook v0 di validazione del mart in `dataset-incubator`.

- scopo: sanity check e lettura base del mart
- non sostituisce l'analisi pubblica
- focus iniziale: pensioni sotto i 500 euro, distribuzione territoriale e differenze di genere


In [ ]:
import duckdb
from pathlib import Path

MART_TABLE = "mart_pensioni_regione_importo"
METRICA = "numero_pensioni"

candidate_dir = Path.cwd()
if not (candidate_dir / "dataset.yml").exists():
    if (candidate_dir.parent / "dataset.yml").exists():
        candidate_dir = candidate_dir.parent
    else:
        raise FileNotFoundError("dataset.yml non trovato nella directory corrente o nella parent.")

mart_root = candidate_dir.parents[1] / "out" / "data" / "mart" / "inps_pensioni_2017_2024"
mart_candidates = sorted(mart_root.glob(f"*/{MART_TABLE}.parquet"))
if not mart_candidates:
    raise FileNotFoundError(f"Mart non trovato sotto {mart_root}. Eseguire prima toolkit run all.")

PARQUET_PATH = mart_candidates[-1]
print(f"Candidate dir: {candidate_dir.name}")
print(f"Parquet path: {PARQUET_PATH}")


In [ ]:
con = duckdb.connect()
df = con.execute("SELECT * FROM read_parquet(?)", [str(PARQUET_PATH)]).df()
print(f"Shape: {df.shape}")
display(df.head(10))


In [ ]:
display(df.dtypes)
display(df.isnull().sum())
print(df[METRICA].describe())


In [ ]:
quota_basse = (
    df[df["classe_importo"] == "Fino a 499,99"]
    .groupby(["anno", "regione"], as_index=False)[METRICA]
    .sum()
)
display(quota_basse.sort_values(["anno", METRICA], ascending=[True, False]).head(20))

gap_genere = (
    df[df["classe_importo"] == "Fino a 499,99"]
    .groupby(["anno", "sesso"], as_index=False)[METRICA]
    .sum()
)
display(gap_genere.sort_values(["anno", "sesso"]))


## Note v0

- Candidate: `inps-pensioni`
- Dataset tecnico: `inps_pensioni_2017_2024`
- Tabella mart: `mart_pensioni_regione_importo`
- Metrica guida: `numero_pensioni`
- Perimetro: regione, sesso, classe età, classe importo, trimestre
- Verifica chiave da fare in lettura: stabilità semantica delle classi di importo lungo la serie 2017-2024
